# 02 — Advanced Telecom RAG demo and evaluation

This notebook builds the upgraded retrieval pipeline and measures each stage separately.

The original baseline was:

```text
question → MiniLM → FAISS top-4 → LLM
```

The upgraded system is:

```text
                         question
                            ↓
                    clean retrieval query
                            ↓
                 ┌──────────┴──────────┐
                 ↓                     ↓
          BGE dense retrieval       BM25 lexical
              top 15                  top 15
                 └──────────┬──────────┘
                            ↓
                  reciprocal-rank fusion
                            ↓
                    ~20 candidates
                            ↓
                  cross-encoder reranker
                            ↓
                       best 4 chunks
                            ↓
                  + full KPI context
                            ↓
                           LLM
```

We evaluate **dense vs hybrid vs reranked retrieval** before evaluating generation. This makes it possible to see where improvements actually come from.

## 0. Environment

Install the pinned environment from the repository root:

```bash
python -m pip install -r requirements-dev.txt
```

The first run downloads two small retrieval models:
- `BAAI/bge-small-en-v1.5` for dense embeddings,
- `cross-encoder/ms-marco-MiniLM-L6-v2` for reranking.

Both run locally; they do not consume LLM API credits.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from telecom_rag.config import (
    EMBEDDING_MODEL, RERANKER_MODEL, CHUNK_SIZE, CHUNK_OVERLAP,
    VECTOR_STORE_DIR, PROCESSED_KPI_PATH,
)

print("Embedding model:", EMBEDDING_MODEL)
print("Reranker:", RERANKER_MODEL)
print("Chunk size / overlap:", CHUNK_SIZE, "/", CHUNK_OVERLAP)

## 1. Download the expanded corpus

The corpus now contains **10 configured sources**, not just four. It combines:
- NR and LTE measurement standards,
- NR data procedures and overall architecture,
- the exact AERPAW/Ericsson experiment documentation,
- applied Ericsson material about beamforming, coverage/capacity, optimization and real-world 5G performance.

A larger corpus makes retrieval harder, which is useful: BM25/reranking now have a meaningful job rather than choosing among only a few documents.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "download_docs.py")],
    check=False,
)

In [ ]:
from telecom_rag.config import DOCS_DIR

source_files = [
    p for p in sorted(DOCS_DIR.glob("*"))
    if p.is_file() and not p.name.startswith("_") and not p.name.startswith(".")
]
print("Downloaded source files:", len(source_files))
for p in source_files:
    print(f"{p.name:48s} {p.stat().st_size / 1024:9.1f} KB")

## 2. Load documents with source metadata

PDFs are loaded page-by-page. For technical PDFs we also attempt to detect a numbered section heading. The metadata later becomes both retrieval context and visible citations.

In [ ]:
from telecom_rag.documents import load_documents

docs = load_documents()
print("Document/page objects:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:600])

## 3. Section-aware contextual chunking

The old baseline split raw text into fixed fragments.

The upgraded chunker:
1. respects paragraphs/newlines before splitting inside sentences,
2. uses larger ~400-token chunks,
3. keeps overlap,
4. prepends document title + section + page to every chunk.

That header is part of the embedding input, so similar definitions from different standards are easier to disambiguate.

In [ ]:
from telecom_rag.documents import chunk_documents

chunks = chunk_documents(docs)
print("Chunks:", len(chunks))
print(chunks[0].metadata)
print(chunks[0].page_content[:800])

## 4. Build the BGE/FAISS index

We switched from a generic MiniLM sentence embedding model to the retrieval-oriented `BAAI/bge-small-en-v1.5`.

The index is versioned with a manifest containing:
- embedding model,
- chunk size,
- overlap,
- chunking version.

This matters because BGE-small and the old MiniLM model are both 384-dimensional. Without a manifest, an old FAISS index could silently be reused with the wrong embedding model.

In [ ]:
from telecom_rag.rag import build_vector_store, index_is_current

store = build_vector_store()
print("Vectors:", store.index.ntotal)
print("Index current:", index_is_current())

## 5. Load the full advanced retriever

The retriever owns:
- FAISS/BGE dense search,
- BM25 lexical search,
- reciprocal-rank fusion (RRF),
- a cross-encoder reranker.

The cross-encoder is loaded lazily only when the `reranked` mode is first used.

In [ ]:
from telecom_rag.rag import load_advanced_retriever

retriever = load_advanced_retriever(build_if_missing=False)
retriever

## 6. Why hybrid retrieval?

Dense search is strong for **meaning**:

> "radio quality under interference"

BM25 is strong for exact telecom/file tokens:

> `NR5G_RSRP`, `Serving_cell_Params_ENDC.csv`, `TS 38.215`

RRF combines the rankings without requiring their raw scores to be on the same scale.

In [ ]:
query = "Which file contains serving-cell RF measurements in the AERPAW workflow?"

for mode in ["dense", "hybrid", "reranked"]:
    result = retriever.retrieve(query, k=4, mode=mode)
    print("\n", "=" * 25, mode.upper(), "=" * 25)
    for i, doc in enumerate(result.documents, 1):
        print(
            i,
            doc.metadata.get("source_id"),
            "page=", doc.metadata.get("page"),
            "methods=", doc.metadata.get("retrieval_methods"),
            "rerank=", doc.metadata.get("rerank_score"),
        )

## 7. Cleaner KPI-conditioned retrieval queries

The previous graph appended the *entire* KPI summary—numbers, percentiles, anomaly scores and timestamps—to the embedding query.

That can dilute retrieval.

The upgraded graph keeps:
- the user's question,
- only useful KPI **names/concepts** for search.

Detailed numeric KPI context is added later, at generation time.

In [ ]:
from telecom_rag.retrieval import build_retrieval_query

example_observation = {
    "nr_rsrp_dbm": -96,
    "nr_sinr_db": 4,
    "nr_cqi": 5,
    "nr_mcs": 7,
    "throughput_mbps": 12,
}
q = "Why might this observation have poor throughput?"
print(build_retrieval_query(q, example_observation))

## 8. Inspect reranked evidence

This is the evidence the LLM will actually see. Always inspect retrieval before changing prompts or models.

In [ ]:
final = retriever.retrieve(
    "Why might reasonable SSB RSRP fail to predict NR downlink throughput?",
    k=4,
    mode="reranked",
)
for i, doc in enumerate(final.documents, 1):
    print(f"\n--- S{i} ---")
    print(doc.metadata)
    print(doc.page_content[:900])

## 9. Choose an LLM

For free/local development:

```bash
ollama pull qwen3:4b
ollama serve
```

Then keep `PROVIDER = "ollama"`.

For a hosted run set `OPENAI_API_KEY` and switch the provider. The retrieval experiment is independent of the generation provider.

In [ ]:
from telecom_rag.rag import get_llm

PROVIDER = "ollama"
MODEL = "qwen3:4b"

llm = get_llm(provider=PROVIDER, model=MODEL)

## 10. LangGraph with the final retrieval pipeline

The graph still has a small purposeful structure:

```text
route question
   ├─ docs-only ──────────────┐
   └─ KPI question            │
          ↓                   │
      analyze KPI             │
          └────────→ retrieve ←┘
                         ↓
                      generate
```

The retrieval node now runs the advanced retriever and stores the clean query for debugging.

In [ ]:
from telecom_rag.graph import build_graph

graph = build_graph(
    llm,
    retriever,
    reference_df=None,
    top_k=4,
    retrieval_mode="reranked",
)

In [ ]:
result = graph.invoke({
    "question": "Why can SSB RSRP be a weak predictor of NR downlink data performance?",
    "use_rag": True,
    "observation": None,
})

print("Route:", result["route"])
print("Retrieval query:", result["retrieval_query"])
print("Retrieval mode:", result["retrieval_mode"])
print("\nAnswer:\n", result["answer"])

## 11. KPI-aware demo

The graph uses numeric KPI percentiles/anomaly context for **generation**, while retrieval stays clean.

In [ ]:
from telecom_rag.data import load_processed_kpis

if PROCESSED_KPI_PATH.exists():
    kpis = load_processed_kpis()
    if "anomaly_score" in kpis.columns:
        selected = kpis.sort_values("anomaly_score", ascending=False).iloc[0]
    else:
        selected = kpis.iloc[0]
    print(selected)
else:
    kpis = None
    selected = None
    print("Run Notebook 01 first for KPI-aware examples.")

In [ ]:
if selected is not None:
    kpi_graph = build_graph(
        llm,
        retriever,
        reference_df=kpis,
        top_k=4,
        retrieval_mode="reranked",
    )
    kpi_result = kpi_graph.invoke({
        "question": (
            "Why might this observation have this throughput, and which radio "
            "measurements should I investigate?"
        ),
        "use_rag": True,
        "observation": selected.to_dict(),
    })
    print("Retrieval query:", kpi_result["retrieval_query"])
    print("\nDetailed KPI generation context:\n", kpi_result.get("kpi_context", ""))
    print("\nAnswer:\n", kpi_result["answer"])

## 12. Improved evaluation set

The original benchmark was dominated by basic definitions that a pretrained LLM already knows.

The new benchmark has **20 questions** split across:
- `general_telecom`,
- `corpus_specific`,
- `standard_specific`,
- `applied_diagnostics`,
- `cross_source`.

This makes RAG earn its value on experiment-specific facts, exact files, applied network-performance explanations and multi-source retrieval.

In [ ]:
from telecom_rag.evaluation import load_eval_questions

questions = load_eval_questions()
eval_df = pd.DataFrame(questions)
print(eval_df.groupby(["category", "difficulty"]).size())
eval_df[["id", "category", "difficulty", "question"]].head(20)

## 13. Retrieval ablation: dense vs hybrid vs reranked

This is the most important diagnostic experiment.

Metrics:
- **source_hit** — at least one expected source was retrieved,
- **source_recall** — fraction of expected source documents retrieved,
- **source_precision** — fraction of retrieved chunks from expected source(s),
- **MRR** — how early the first expected source appeared.

If reranking does not beat dense/hybrid retrieval, inspect those failures before touching the LLM.

In [ ]:
from telecom_rag.evaluation import evaluate_retrieval, summarize_retrieval

retrieval_results = evaluate_retrieval(
    retriever,
    questions,
    k=4,
    modes=("dense", "hybrid", "reranked"),
)
retrieval_summary = summarize_retrieval(retrieval_results)
retrieval_summary

In [ ]:
retrieval_by_category = (
    retrieval_results
    .groupby(["category", "mode"])[["source_hit", "source_recall", "mrr"]]
    .mean()
    .round(3)
)
retrieval_by_category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metric = "source_recall"
summary_for_plot = retrieval_summary[metric]
ax.bar(summary_for_plot.index, summary_for_plot.values)
ax.set_ylim(0, 1)
ax.set_ylabel(metric)
ax.set_title("Retrieval ablation: expected-source recall@4")
plt.tight_layout()
plt.show()

## 14. Inspect retrieval failures

This tells us whether remaining errors are:
- semantic retrieval failures,
- lexical retrieval failures,
- reranker failures,
- benchmark/source-label issues.

In [ ]:
failed = retrieval_results[
    (retrieval_results["mode"] == "reranked") &
    (retrieval_results["source_recall"] < 1)
]
failed[[
    "id", "category", "question",
    "expected_source_ids", "retrieved_source_ids",
    "source_recall", "mrr"
]]

## 15. Same LLM: baseline vs final RAG

Only now do we test generation.

The comparison uses the same LLM and questions. RAG uses the final reranked pipeline.

Generation metrics:
- semantic similarity to a short reference answer,
- required-fact recall,
- citation presence,
- citation validity,
- answer-to-context similarity as a simple transparent grounding proxy,
- latency.

The context-similarity metric is **not an entailment/factuality score**; it should be interpreted together with citations and manual failure inspection.

In [ ]:
from telecom_rag.rag import get_embeddings
from telecom_rag.evaluation import compare_baseline_and_rag, summarize_comparison

embeddings = get_embeddings()

results = compare_baseline_and_rag(
    llm=llm,
    retriever=retriever,
    embeddings=embeddings,
    questions=questions,
    k=4,
    retrieval_mode="reranked",
    limit=10,  # set None for all 20
)

summarize_comparison(results)

## 16. The important view: results by category

A basic telecom definition is where the LLM-only baseline is strongest.

The RAG system should show its clearest advantage on:
- corpus-specific,
- standard-specific,
- applied-diagnostic,
- cross-source questions.

In [ ]:
category_summary = summarize_comparison(results, by_category=True)
category_summary

In [ ]:
pivot = (
    results.groupby(["category", "mode"])["required_term_recall"]
    .mean()
    .unstack("mode")
)
pivot.plot(kind="bar", figsize=(9, 5))
plt.ylim(0, 1)
plt.ylabel("Required-fact recall")
plt.title("LLM-only vs RAG by question category")
plt.tight_layout()
plt.show()

## 17. Inspect answer-level failures

Averages can hide the useful story. For each bad case ask:

1. Was the expected source retrieved?
2. Did the reranker place it in the final top-4?
3. Did the answer use/cite the evidence?
4. Did the benchmark require wording that is too brittle?
5. Did RAG correctly say the evidence was insufficient?

In [ ]:
for qid in results["id"].unique():
    pair = results[results["id"] == qid]
    print("\n" + "=" * 90)
    print(qid, pair.iloc[0]["category"], "-", pair.iloc[0]["question"])
    for _, row in pair.iterrows():
        print(f"\n[{row['mode']}] term_recall={row['required_term_recall']:.2f} "
              f"semantic={row['semantic_similarity']:.2f}")
        print(row["answer"][:1000])

## 18. What to tune next

Do not blindly add more agents or a larger LLM.

Use the ablations:
- low dense recall → embedding/query/chunk problem,
- hybrid beats dense → exact terminology matters,
- reranker beats hybrid → candidate retrieval is good but ordering was weak,
- retrieval is strong but answer is weak → prompt/generator issue,
- generic questions show little gain but corpus-specific questions improve → expected RAG behavior.

Useful next controlled experiments:
- chunk size: 1200 / 1700 / 2200 characters,
- candidate counts: 10 / 20 / 30,
- final top-k: 3 / 4 / 6,
- BGE small vs base,
- reranker on/off,
- corpus subsets (standards only vs standards + applied Ericsson material).